<img src="https://courseware.decoded.com/images/decoded/logo-decoded.png" align="left" width="100px"/>

# Regression model selection and validation

**Learner notebook — standalone version**

In this notebook, you will build and compare several regression models for bike-rental demand. You will use cross-validation, tune hyperparameters and evaluate the final model on unseen data.

---

<small>© COPYRIGHT 2026 DECODED LIMITED. ALL RIGHTS RESERVED. Web: https://decoded.com/</small>

## Why this matters

A regression model predicts a **continuous numerical value**. In this scenario, the target is the number of bikes rented during a given period.

A useful model could help an operations team:

- plan bike availability
- schedule maintenance and redistribution
- anticipate demand during different weather conditions
- make staffing decisions

The aim is not to find a model that performs well only on historical data. We want a model that can make useful predictions for new, unseen observations.

## Learning outcomes

By the end of this notebook, you will be able to:

1. prepare data for a regression task
2. establish a simple baseline
3. train linear and tree-based regression algorithms
4. compare models using cross-validation
5. explain why MAE, RMSE and $R^2$ provide different information
6. tune model hyperparameters without using the test set
7. evaluate a final model on unseen data
8. interpret model errors in a workplace context

## Learning journey

1. Load and inspect the data  
2. Define the feature matrix and target  
3. Create a holdout test set  
4. Build preprocessing pipelines  
5. Establish a baseline  
6. Compare different regression algorithms  
7. Interpret regression metrics  
8. Tune the strongest candidate model  
9. Evaluate the final model  
10. Reflect on limitations and next steps

## 1. Setup

Run this cell first. It imports the libraries used throughout the notebook and sets a fixed random seed so that results are reproducible.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_regression
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

print("Setup complete.")

## 2. Load the dataset

The notebook first looks for a Seoul bike-rental CSV in common notebook locations.

To keep this notebook runnable as a standalone learning resource, it creates a clearly labelled synthetic bike-demand dataset when the CSV is not available. The synthetic data is for practice only and should not be used for operational decisions.

In [ ]:
candidate_paths = [
    Path("seoulbikerentals.csv"),
    Path("/mnt/data/seoulbikerentals.csv"),
    Path("SeoulBikeData.csv"),
    Path("/mnt/data/SeoulBikeData.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is not None:
    data = pd.read_csv(data_path, encoding_errors="ignore")
    print(f"Loaded dataset from: {data_path}")
else:
    rng = np.random.default_rng(RANDOM_STATE)
    n_rows = 1200

    hour = rng.integers(0, 24, n_rows)
    temperature = rng.normal(13, 9, n_rows)
    humidity = np.clip(rng.normal(58, 18, n_rows), 10, 100)
    wind_speed = np.clip(rng.normal(2.2, 1.0, n_rows), 0, None)
    rainfall = np.where(rng.random(n_rows) < 0.12, rng.gamma(1.5, 3.0, n_rows), 0)
    season = rng.choice(["Spring", "Summer", "Autumn", "Winter"], n_rows)
    holiday = rng.choice(["No Holiday", "Holiday"], n_rows, p=[0.93, 0.07])
    functioning_day = rng.choice(["Yes", "No"], n_rows, p=[0.97, 0.03])

    commute_peak = np.where(
        ((hour >= 7) & (hour <= 9)) | ((hour >= 16) & (hour <= 19)),
        280,
        0,
    )
    seasonal_effect = pd.Series(season).map(
        {"Spring": 80, "Summer": 140, "Autumn": 110, "Winter": -90}
    ).to_numpy()

    demand = (
        320
        + 18 * temperature
        - 3.0 * humidity
        - 32 * rainfall
        + commute_peak
        + seasonal_effect
        - np.where(holiday == "Holiday", 100, 0)
        - np.where(functioning_day == "No", 500, 0)
        + rng.normal(0, 95, n_rows)
    )
    demand = np.clip(demand, 0, None).round().astype(int)

    data = pd.DataFrame(
        {
            "hour": hour,
            "temperature_c": temperature.round(1),
            "humidity_pct": humidity.round(1),
            "wind_speed_m_s": wind_speed.round(1),
            "rainfall_mm": rainfall.round(1),
            "season": season,
            "holiday": holiday,
            "functioning_day": functioning_day,
            "rented_bike_count": demand,
        }
    )
    print("CSV not found. Using the synthetic bike-demand practice dataset.")

print("Dataset shape:", data.shape)
data.head()

### Check the column names

Real datasets often use different capitalisation, spaces or punctuation. The next cell standardises column names so they are easier to reference in Python.

In [ ]:
data.columns = (
    data.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

data.columns.tolist()

### Identify the target

The intended target is `rented_bike_count`. The cell also checks for common versions of the same column name and stops with a clear message when no suitable target is found.

In [ ]:
target_candidates = [
    "rented_bike_count",
    "rented_bike_count_hour",
    "rented_bike_count_count",
]

target_column = next(
    (column for column in target_candidates if column in data.columns),
    None,
)

if target_column is None:
    possible_targets = [
        column for column in data.columns
        if "rent" in column and "bike" in column
    ]
    if len(possible_targets) == 1:
        target_column = possible_targets[0]
    else:
        raise ValueError(
            "A bike-rental target column could not be identified. "
            "Rename the numerical target to 'rented_bike_count'."
        )

print("Target column:", target_column)

## 3. Inspect data quality

Before modelling, check:

- data types
- missing values
- whether the target is numerical
- the scale and distribution of the target

This is a focused modelling notebook, so it does not attempt a complete exploratory analysis.

In [ ]:
summary = pd.DataFrame(
    {
        "data_type": data.dtypes.astype(str),
        "missing_values": data.isna().sum(),
        "missing_pct": (data.isna().mean() * 100).round(1),
        "unique_values": data.nunique(),
    }
)

summary

In [ ]:
if not pd.api.types.is_numeric_dtype(data[target_column]):
    data[target_column] = pd.to_numeric(data[target_column], errors="coerce")

data = data.dropna(subset=[target_column]).copy()

data[target_column].describe().to_frame("target_summary")

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(data[target_column], bins=30)
plt.xlabel("Rented bike count")
plt.ylabel("Number of observations")
plt.title("Distribution of the regression target")
plt.show()

### Learner checkpoint

Discuss the following:

1. Is the target strongly skewed?
2. Are unusually high or low values likely to matter operationally?
3. Which errors would be more costly: under-predicting or over-predicting demand?

## 4. Define features and target

- `X` contains the information available to the model.
- `y` contains the numerical value the model will learn to predict.

Columns that directly reveal the target should never be included in `X`.

In [ ]:
X = data.drop(columns=[target_column])
y = data[target_column]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
X.head()

## 5. Create a holdout test set

The test set is kept separate until the end. It acts as a final check of how the chosen model performs on unseen data.

We do **not** use the test set to choose an algorithm or tune hyperparameters. Repeatedly checking it during development would make it part of the training process and produce an over-optimistic estimate.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 6. Build a preprocessing pipeline

A pipeline keeps preprocessing and modelling together.

This matters because every cross-validation fold must learn its preprocessing steps from that fold's training portion only. Performing preprocessing on the full dataset before cross-validation can leak information from validation rows into training.

The pipeline:

- fills missing numerical values with the median
- standardises numerical variables
- fills missing categorical values with the most frequent category
- one-hot encodes categorical variables

In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

## 7. Regression metrics: why use more than one?

No single regression metric tells the whole story.

| Metric | What it measures | Why it is useful | Limitation |
|---|---|---|---|
| **MAE** | Average absolute prediction error | Easy to explain in the target's original unit; every error contributes in direct proportion to its size | Does not emphasise occasional large errors |
| **RMSE** | Square root of the average squared error | Gives larger errors more influence, which is useful when severe under- or over-predictions are costly | Can be dominated by outliers |
| **$R^2$** | Improvement over predicting the mean | Shows how much variation the model explains relative to a simple mean baseline | Does not show the typical error in bike counts and can be negative |
| **MAPE** | Average absolute percentage error | Can be intuitive for relative error | Unstable or undefined when actual values are zero or close to zero |

For this dataset:

- use **MAE** to communicate a typical error in bike counts
- use **RMSE** when large forecasting misses matter operationally
- use **$R^2$** as a relative measure of fit
- avoid relying on MAPE because demand may be zero

## 8. Establish a baseline

A sophisticated model should outperform a simple reference point.

`DummyRegressor` predicts the mean training target for every observation. It does not learn relationships between features and demand. If another model cannot beat this baseline, it is not adding useful predictive value.

In [ ]:
baseline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DummyRegressor(strategy="mean")),
    ]
)

baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

baseline_metrics = {
    "model": "Mean baseline",
    "MAE": mean_absolute_error(y_test, baseline_predictions),
    "RMSE": mean_squared_error(y_test, baseline_predictions) ** 0.5,
    "R2": r2_score(y_test, baseline_predictions),
}

pd.DataFrame([baseline_metrics]).round(3)

## 9. Compare different regression algorithms

We will compare models with different assumptions:

- **Linear Regression** — a transparent linear benchmark
- **Ridge Regression** — a regularised linear model that can reduce unstable coefficients
- **Decision Tree** — captures non-linear rules but can overfit
- **Random Forest** — averages many trees to improve stability
- **Gradient Boosting** — builds trees sequentially to correct earlier errors

The purpose is not to assume that the most complex model is best. We compare them under the same validation process.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=None,
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=150,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=RANDOM_STATE,
    ),
}

pipelines = {
    name: Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    for name, model in models.items()
}

list(pipelines)

## 10. Cross-validation

A single train/test split can produce a result that depends heavily on which observations happen to be selected.

Five-fold cross-validation:

1. divides the training data into five parts
2. trains on four parts
3. validates on the remaining part
4. repeats until every part has been used for validation
5. summarises the model's average performance and variation

We use the training set only. The holdout test set remains untouched.

In [ ]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

cv_rows = []

for model_name, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    cv_rows.append(
        {
            "model": model_name,
            "mean_MAE": -scores["test_mae"].mean(),
            "std_MAE": scores["test_mae"].std(),
            "mean_RMSE": -scores["test_rmse"].mean(),
            "std_RMSE": scores["test_rmse"].std(),
            "mean_R2": scores["test_r2"].mean(),
            "std_R2": scores["test_r2"].std(),
        }
    )

cv_results = (
    pd.DataFrame(cv_rows)
    .sort_values("mean_RMSE")
    .reset_index(drop=True)
)

cv_results.round(3)

### Why are MAE and RMSE negative in scikit-learn scoring?

Scikit-learn follows the rule that **larger scores are better**. Errors such as MAE and RMSE are losses, where smaller values are better, so the cross-validation API returns their negative values.

The code changes the sign back before reporting the results:

```python
mean_rmse = -scores["test_rmse"].mean()
```

The reported RMSE is therefore a normal positive error value.

In [ ]:
plot_data = cv_results.sort_values("mean_RMSE", ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(plot_data["model"], plot_data["mean_RMSE"], xerr=plot_data["std_RMSE"])
plt.xlabel("Cross-validated RMSE")
plt.ylabel("Model")
plt.title("Model comparison: lower RMSE is better")
plt.show()

### Interpret the model comparison

Use both the average score and its variation.

- A lower average RMSE suggests stronger predictive performance.
- A smaller standard deviation suggests more consistent performance across folds.
- A small difference between two models may not be practically meaningful.
- A more complex model may not be the best choice when a simpler model performs almost as well and is easier to explain.

**Learner task:** Identify the strongest candidate model. Write one sentence that uses both its average RMSE and its fold-to-fold variation.

## 11. Hyperparameter tuning

A model parameter is learned from the data. A hyperparameter is chosen before training.

For a random forest:

- `n_estimators` controls the number of trees
- `max_depth` limits how deep each tree can grow
- `min_samples_split` controls how many observations are required to split a node
- `max_features` controls how many features each tree considers at a split

We use `GridSearchCV` to test combinations under cross-validation.

The grid search is fitted to the **training set only**. The test set is still protected.

In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

parameter_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 8, 16],
    "model__min_samples_split": [2, 5],
    "model__max_features": ["sqrt", 1.0],
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=parameter_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    refit=True,
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)
print()
print(f"Best cross-validated RMSE: {-grid_search.best_score_:.3f}")

### Review more than the winning combination

The best score alone does not show whether several combinations performed similarly. The next table helps you compare the strongest settings and decide whether extra model complexity produces a worthwhile improvement.

In [ ]:
tuning_results = pd.DataFrame(grid_search.cv_results_)

tuning_summary = (
    tuning_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "param_model__n_estimators",
            "param_model__max_depth",
            "param_model__min_samples_split",
            "param_model__max_features",
        ]
    ]
    .assign(
        mean_CV_RMSE=lambda frame: -frame["mean_test_score"]
    )
    .sort_values("rank_test_score")
    .drop(columns="mean_test_score")
    .head(10)
)

tuning_summary.round(3)

## 12. Final evaluation on unseen data

Only now do we use the holdout test set.

The selected pipeline already includes preprocessing and the tuned random forest. `GridSearchCV` refits the best configuration on the full training set, so it is ready to make test predictions.

In [ ]:
final_model = grid_search.best_estimator_
test_predictions = final_model.predict(X_test)

final_metrics = pd.DataFrame(
    [
        {
            "model": "Mean baseline",
            "MAE": mean_absolute_error(y_test, baseline_predictions),
            "RMSE": mean_squared_error(y_test, baseline_predictions) ** 0.5,
            "R2": r2_score(y_test, baseline_predictions),
        },
        {
            "model": "Tuned Random Forest",
            "MAE": mean_absolute_error(y_test, test_predictions),
            "RMSE": mean_squared_error(y_test, test_predictions) ** 0.5,
            "R2": r2_score(y_test, test_predictions),
        },
    ]
)

final_metrics.round(3)

### Read the final metrics in context

A suitable interpretation might be:

> The tuned model's MAE estimates the typical difference between predicted and actual demand in bike counts. RMSE is higher because it gives more weight to larger misses. The $R^2$ score shows how much better the model explains variation in demand than predicting the mean alone.

Avoid saying that an $R^2$ of 0.80 means the model is “80% accurate”. Regression accuracy is not represented by one percentage.

## 13. Diagnose the errors

Metrics summarise performance, but plots can reveal patterns that a single number hides.

### Actual versus predicted

A strong model should place observations reasonably close to the diagonal reference line.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, test_predictions, alpha=0.6)

minimum = min(y_test.min(), test_predictions.min())
maximum = max(y_test.max(), test_predictions.max())
plt.plot([minimum, maximum], [minimum, maximum], linestyle="--")

plt.xlabel("Actual bike demand")
plt.ylabel("Predicted bike demand")
plt.title("Actual versus predicted demand")
plt.show()

### Residual plot

A residual is:

```text
actual value - predicted value
```

- a positive residual means the model under-predicted demand
- a negative residual means the model over-predicted demand
- a visible pattern may suggest that the model is systematically missing part of the relationship

In [ ]:
residuals = y_test - test_predictions

plt.figure(figsize=(8, 5))
plt.scatter(test_predictions, residuals, alpha=0.6)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted bike demand")
plt.ylabel("Residual")
plt.title("Residuals versus predictions")
plt.show()

### Inspect the largest errors

Large errors can be more useful than average performance when deciding whether a model is safe or operationally useful.

In [ ]:
error_review = X_test.copy()
error_review["actual_demand"] = y_test
error_review["predicted_demand"] = test_predictions
error_review["residual"] = residuals
error_review["absolute_error"] = np.abs(residuals)

error_review.sort_values("absolute_error", ascending=False).head(10)

### Learner task

Review the ten largest errors.

1. Do the largest errors share any visible conditions?
2. Is the model more likely to under-predict or over-predict these cases?
3. What additional feature could help explain the errors?
4. What would be the operational impact of these mistakes?

## 14. Compare training and test performance

A large gap between training and test error can indicate overfitting.

This check is not a replacement for cross-validation, but it helps diagnose whether the final model has learned the training data too closely.

In [ ]:
train_predictions = final_model.predict(X_train)

generalisation_check = pd.DataFrame(
    [
        {
            "dataset": "Training",
            "MAE": mean_absolute_error(y_train, train_predictions),
            "RMSE": mean_squared_error(y_train, train_predictions) ** 0.5,
            "R2": r2_score(y_train, train_predictions),
        },
        {
            "dataset": "Test",
            "MAE": mean_absolute_error(y_test, test_predictions),
            "RMSE": mean_squared_error(y_test, test_predictions) ** 0.5,
            "R2": r2_score(y_test, test_predictions),
        },
    ]
)

generalisation_check.round(3)

## 15. Responsible interpretation

A good numerical score does not guarantee that a model is ready for use.

Before deployment, consider:

- **Representativeness:** Does the training data reflect the locations, seasons and operating conditions where predictions will be used?
- **Data leakage:** Are any features only known after demand has occurred?
- **Concept drift:** Could travel patterns, pricing or infrastructure change over time?
- **Error impact:** What happens when demand is under-predicted?
- **Human oversight:** Who reviews unusual forecasts?
- **Monitoring:** Which metrics will be tracked after deployment?
- **Purpose:** Is the model supporting a decision, or making it automatically?

Model evaluation is both a technical and a business judgement.

## 16. Knowledge check

1. Why do we keep a holdout test set?
2. Why should preprocessing happen inside a pipeline?
3. What does cross-validation tell us that one validation split cannot?
4. Why might RMSE be more useful than MAE when large errors are costly?
5. Why can $R^2$ be misleading on its own?
6. What is the difference between a model parameter and a hyperparameter?
7. Why must the test set remain outside `GridSearchCV`?
8. What evidence would you need before using this model operationally?

## Key takeaways

- Start with a baseline so that model performance has a meaningful reference point.
- Compare different algorithms using the same cross-validation strategy.
- Use pipelines to keep preprocessing inside each training fold and reduce leakage.
- Use MAE, RMSE and $R^2$ together because they answer different questions.
- Tune hyperparameters on training data, not on the test set.
- Use the holdout test set once for a final estimate of generalisation.
- Investigate individual errors as well as average metrics.
- Choose a model based on predictive performance, stability, explainability and operational impact.

## What to learn next

Next, explore:

- feature engineering for time, weather and calendar variables
- time-aware validation when observations are ordered chronologically
- model explainability and feature importance
- prediction intervals and uncertainty
- monitoring model performance after deployment
- comparing regression workflows with classification workflows

## Technical notes

Packages used:

- `pandas`
- `numpy`
- `matplotlib`
- `scikit-learn`

The notebook is designed for scikit-learn versions that support `OneHotEncoder(handle_unknown="ignore")` and the `neg_root_mean_squared_error` scoring name.